# Trees and Ensembles, Notebook 1
## Entropy and information: measuring how much we know and how much we learn

**Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### What this is about

Before building a decision tree we have to answer a question that sounds philosophical
and turns out to be very practical: **how do you measure how much you know?** If a bank has 14 clients and
8 of them paid, how much uncertainty is there about the next one? And if we also know that the client has
little debt, how much of that uncertainty goes away?

Claude Shannon answered that in 1948 with a number: **entropy**. And the reduction in
entropy when we learn a piece of data, **mutual information** (or information gain), turns out to be
exactly what a decision tree uses to choose which question to ask first.

This notebook uses a table of 14 firms so that every calculation can be checked by hand,
and then simulations to see what 14 rows cannot show: what happens when the
experiment is repeated many times.

### What you will learn today

1. What the information of an event is and why it is measured with a logarithm.
2. What the entropy of a variable is and how to compute it from counts.
3. What conditional entropy and information gain (mutual information) are.
4. How they relate to what you already know: entropy is a relative of **variance**
   and mutual information is a relative of **correlation**, with one important difference:
   correlation only sees straight-line relationships; mutual information sees any relationship.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("Ready.")


## 1. The information of an event: surprise

A certain event tells us nothing. A rare one tells us a lot. Shannon put a number on that intuition:

$$
I(\text{event}) = -\log_2(p)
$$

where $p$ is the probability of the event and the result is measured in **bits**. A fair coin
($p = 0.5$) carries 1 bit; a die ($p = 1/6$) carries 2.58 bits; a certain event ($p = 1$),
0 bits.

**Why the logarithm.** So that two independent events *add* their information:
$-\log_2(p \cdot q) = -\log_2(p) - \log_2(q)$. Two coins carry 2 bits, not "0.25 of something".
The logarithm is the only function with that property.


In [ ]:

eventos = pd.DataFrame({
    "evento": ["heads (fair coin)", "rolling a 3 (die)", "17 comes up (36-slot roulette)",
               "an AAA-rated firm defaults this year", "a C-rated firm defaults this year", "the sun rises tomorrow"],
    "p": [0.5, 1/6, 1/36, 0.01, 0.6, 1.0],
})
eventos["informacion_bits"] = -np.log2(eventos["p"])
eventos.round(3)


In [ ]:

p = np.linspace(0.01, 1, 200)
fig = px.line(x=p, y=-np.log2(p), labels={"x": "probability p", "y": "information, in bits"},
              title="Information of an event as a function of its probability: −log₂(p)")
fig.show()


## 2. Entropy: average surprise

If each value of a variable carries $-\log_2 p(v)$ of information, the whole variable carries, on average:

$$
H(X) = -\sum_{v} p(v) \, \log_2 p(v)
$$

What each piece means: $v$ runs over the possible values of $X$ and $p(v)$ is the fraction of
times each one occurs. $H$ reads as "how many yes/no questions are needed, on
average, to guess the value". A fair coin: 1 bit. A coin loaded to 90%: 0.47
bits (we almost always know what is coming). A constant: 0.

Let's see it on the 14 firms from the course.


In [ ]:

# The 14 firms that applied for credit (the small table from the course)
credito = pd.DataFrame({
    "tamano":    ["pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande"],
    "deuda":     ["baja", "baja", "baja", "baja", "alta", "alta", "alta", "alta", "alta", "media", "media", "media", "media", "media"],
    "garantia":  ["no", "si", "no", "si", "si", "si", "no", "no", "no", "no", "si", "si", "no", "no"],
    "historial": ["bueno", "malo", "malo", "bueno", "bueno", "malo", "bueno", "malo", "bueno", "bueno", "bueno", "malo", "malo", "malo"],
    "paga":      ["si", "si", "si", "si", "si", "si", "no", "no", "no", "si", "si", "no", "no", "no"],
})
credito.index = range(1, 15)
credito


In [ ]:

def entropia(serie):
    # Entropy in bits of a column: count, convert to probabilities, sum -p log2 p
    p = serie.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

for columna in credito.columns:
    print(f"H({columna:9s}) = {entropia(credito[columna]):.3f} bits    values: {dict(credito[columna].value_counts())}")


**How to read it.** 8 of the 14 firms pay, so $p(\text{si}) = 0.571$ and
$H(\text{paga}) = 0.985$ bits: almost the maximum of 1 bit. Guessing whether a firm pays is almost
a fair coin. The variables with three values (size, debt) can go up to
$\log_2 3 = 1.585$ bits.

The full curve for a two-valued variable: entropy is highest when the two
options are equally likely and drops to zero at the extremes.


In [ ]:

p = np.linspace(0.001, 0.999, 300)
H = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
fig = px.line(x=p, y=H, labels={"x": "p(si)", "y": "H, in bits"}, title="Entropy of a two-valued variable")
p_paga = (credito["paga"] == "si").mean()
fig.add_trace(go.Scatter(x=[p_paga], y=[entropia(credito["paga"])], mode="markers+text", text=["paga"], textposition="top center",
                         marker=dict(size=12, color="red"), name="paga in the 14 firms"))
fig.show()


## 3. Conditional entropy and gain: how much I learn by asking

If I know a firm's debt level, how much uncertainty is left about whether it pays? We
compute the entropy of "paga" **within each debt group** and average, weighting each
group by its size:

$$
H(Y \mid X) = \sum_{v} P(X = v) \cdot H(Y \text{ within the group } X = v)
$$

And the **information gain**, also called **mutual information**, is what disappears:

$$
I(Y; X) = H(Y) - H(Y \mid X)
$$

Here $Y$ is the variable we want to predict (paga) and $X$ the attribute we are considering
asking about. An attribute with high gain is a good question; with zero gain, a
useless question.


In [ ]:

def entropia_condicional(df, objetivo, atributo):
    # Weighted average of the target entropy within each group of the attribute
    total = 0.0
    for valor, grupo in df.groupby(atributo):
        peso = len(grupo) / len(df)
        total += peso * entropia(grupo[objetivo])
    return total

def ganancia(df, objetivo, atributo):
    return entropia(df[objetivo]) - entropia_condicional(df, objetivo, atributo)

# The detail for debt, group by group 
print("H(paga) =", round(entropia(credito["paga"]), 3))
for valor, grupo in credito.groupby("deuda"):
    print(f"  deuda = {valor:6s}: {len(grupo)} firms, {int((grupo.paga == 'si').sum())} pay, "
          f"H = {entropia(grupo.paga):.3f}, weight {len(grupo)/14:.3f}, contributes {len(grupo)/14 * entropia(grupo.paga):.3f}")
print("H(paga | deuda) =", round(entropia_condicional(credito, "paga", "deuda"), 3))
print("gain(deuda) =", round(ganancia(credito, "paga", "deuda"), 3), "bits")


In [ ]:

tabla = pd.DataFrame({
    "atributo": ["tamano", "deuda", "garantia", "historial"],
})
tabla["H(paga)"] = entropia(credito["paga"])
tabla["H(paga | X)"] = [entropia_condicional(credito, "paga", a) for a in tabla["atributo"]]
tabla["ganancia"] = tabla["H(paga)"] - tabla["H(paga | X)"]
tabla["H(X)"] = [entropia(credito[a]) for a in tabla["atributo"]]
tabla["gain ratio"] = tabla["ganancia"] / tabla["H(X)"]
tabla.round(3)


**What the table says.** Debt wins with 0.292 bits: asking about debt leaves almost a
third less uncertainty. Size contributes almost nothing (0.006). That table is, literally,
the first decision of a tree: the root is the attribute with the highest gain. We will build it
in Notebook 2.

**The gain ratio and the identifier trap.** An attribute with many values chops the table
into tiny pieces and gains "for free". The extreme case is an identifier: one value per firm,
$H(\text{paga} \mid \text{id}) = 0$, maximum gain and useless (a new firm brings an id we
have never seen). C4.5 divides the gain by the entropy of the attribute, $H(X)$, to penalize that.


In [ ]:

con_id = credito.copy()
con_id["id"] = range(1, 15)
g = ganancia(con_id, "paga", "id"); h = entropia(con_id["id"])
print(f"gain(id) = {g:.3f} bits (the maximum possible)   H(id) = {h:.3f}   gain ratio = {g/h:.3f}")
print("The gain ratio of id (0.259) still beats that of debt (0.185) in a 14-row table.")
print("That is why C4.5 also requires the gain to exceed the average, and why common sense rules:")
print("an attribute that identifies rows cannot generalize.")


## 4. The parallel with the statistics you already know

### Entropy and variance

Variance measures how much a **number** spreads around its mean. Entropy measures
how much uncertainty a variable **of any type** has, and it has no units. Both
are measures of dispersion, and both are zero when the variable is constant. But:

- entropy only looks at the **probabilities**: three equally likely classes are worth $\log_2 3$ bits, whether
  they are 1, 2, 3 or 10, 20, 30;
- variance looks at the **values** and their units: going from 1, 2, 3 to 10, 20, 30 multiplies
  it by 100.

That is why entropy works for variables without numbers (pays yes/no, size small/medium/large),
where variance makes no sense. And that is why a **classification** tree uses entropy or Gini
exactly where a **regression** tree uses variance: as a measure of impurity that a good
split reduces.


In [ ]:

def entropia_de_probabilidades(p):
    p = np.array(p); p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def varianza_discreta(valores, p):
    valores = np.array(valores, dtype=float); p = np.array(p)
    media = (valores * p).sum()
    return float((p * (valores - media) ** 2).sum())

distribuciones = [
    ("constant (5)",                     [5],                [1.0]),
    ("fair coin (0 or 1)",               [0, 1],             [0.5, 0.5]),
    ("loaded coin (0.9 / 0.1)",          [0, 1],             [0.9, 0.1]),
    ("fair die (1 to 6)",                [1, 2, 3, 4, 5, 6], [1/6] * 6),
    ("three equal classes (1, 2, 3)",    [1, 2, 3],          [1/3] * 3),
    ("three equal classes (10, 20, 30)", [10, 20, 30],       [1/3] * 3),
]
pd.DataFrame([(n, len(v), entropia_de_probabilidades(p), varianza_discreta(v, p)) for n, v, p in distribuciones],
             columns=["distribution", "states", "entropy (bits)", "variance"]).round(3)


For a **continuous** variable, entropy needs the values grouped into bins (like a
histogram), and the result depends on how many bins are used. Variance does not. An
experiment: four variables with 2,000 values each, grouped into 10 equal-width bins.


In [ ]:

def entropia_continua(x, clases=10):
    # Bins into equal-width classes between the minimum and the maximum and computes H from the frequencies
    conteos, _ = np.histogram(x, bins=clases)
    return entropia_de_probabilidades(conteos / conteos.sum())

rng = np.random.default_rng(7)
n = 2000
muestras = {
    "normal(0, 1)":        rng.normal(0, 1, n),
    "normal(0, 3)":        rng.normal(0, 3, n),
    "uniform(-1.7, 1.7)":  rng.uniform(-1.7, 1.7, n),   # same variance as normal(0, 1)
    "lognormal(0, 1)":     rng.lognormal(0, 1, n),
}
pd.DataFrame({nombre: {"variance": np.var(x), "standard deviation": np.std(x),
                       "entropy (10 bins)": entropia_continua(x, 10), "entropy (30 bins)": entropia_continua(x, 30)}
              for nombre, x in muestras.items()}).T.round(3)


**What to look at.** normal(0, 1) and normal(0, 3) have almost the same entropy (the shape is the same,
only the scale changes) and variances that differ by a factor of 9. The uniform has the same variance as
normal(0, 1) and more entropy: it spreads its values more evenly across the bins. The lognormal has
a large variance because of its long tail and little entropy: almost everything falls in the low bins.
Entropy measures *how spread out* the variable is; variance, *how far* it reaches. And with
30 bins instead of 10 all the entropies go up: for continuous variables the number depends
on how you bin.

### Mutual information and correlation

Pearson correlation measures how much the relationship between $X$ and $Y$ resembles a **straight line**
(from $-1$ to $1$). Mutual information measures how much $X$ reduces the uncertainty of $Y$, **whatever
the shape** of the relationship (in bits, from 0 upward). With the variables grouped
into bins:

$$
I(X; Y) = \sum_{x, y} p(x, y) \, \log_2 \frac{p(x, y)}{p(x) \, p(y)}
$$

where $p(x, y)$ is the fraction of observations in the cell (bin of $X$, bin of $Y$) and
$p(x)$, $p(y)$ the fractions of each bin separately. If $X$ and $Y$ were independent,
$p(x, y) = p(x) \, p(y)$ in every cell and the sum would be zero.

The decisive experiment: three simulated relationships, a straight line, a **U** and pure chance.


In [ ]:

def informacion_mutua(x, y, clases=3):
    # Bins x and y into equal-width classes and computes I(X; Y) from the joint table
    cx = pd.cut(x, bins=clases, labels=False)
    cy = pd.cut(y, bins=clases, labels=False)
    conjunta = pd.crosstab(cx, cy, normalize=True).values
    px = conjunta.sum(axis=1, keepdims=True); py = conjunta.sum(axis=0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        aportes = np.where(conjunta > 0, conjunta * np.log2(conjunta / (px * py)), 0.0)
    return float(aportes.sum())

rng = np.random.default_rng(3)
n = 300
x = rng.uniform(0, 1, n)
ruido = rng.normal(0, 0.05, n)
simulado = pd.DataFrame({
    "x": x,
    "y_recta": 0.3 + 0.55 * x + ruido,
    "y_U": 0.1 + 0.9 * (x - 0.5) ** 2 + ruido,
    "y_azar": 0.2 + 0.6 * rng.uniform(0, 1, n) + ruido,
})
fig = px.scatter(simulado.melt(id_vars="x", var_name="relación", value_name="y"), x="x", y="y", facet_col="relación", opacity=0.6,
                 title="Three relationships: a straight line, a U and chance")
fig.show()

resumen = pd.DataFrame({
    "correlation": [simulado["x"].corr(simulado[c]) for c in ["y_recta", "y_U", "y_azar"]],
    "mutual information (bits, 3 bins)": [informacion_mutua(simulado["x"], simulado[c], 3) for c in ["y_recta", "y_U", "y_azar"]],
    "mutual information (bits, 5 bins)": [informacion_mutua(simulado["x"], simulado[c], 5) for c in ["y_recta", "y_U", "y_azar"]],
}, index=["line", "U", "chance"])
resumen.round(3)


**What the table says.** The line: both measures see it. The U: correlation stays close to
zero because the rising half cancels the falling half, but mutual information is
clearly higher than that of chance. Knowing $x$ **does** say a lot about $y$; what is missing is a
straight line. Chance: correlation and mutual information close to zero (mutual information is never
exactly zero with finite data: the counts are noisy, and that is the floor against which
you have to compare).

**Why it matters for what comes next.** A tree chooses its splits by information gain
(or Gini, or variance), without assuming any shape. With two splits on $x$ it finds the U.
A linear regression on $x$ ignores it completely. Let's test it.


In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

X_ = simulado[["x"]]
for objetivo in ["y_recta", "y_U"]:
    lineal = LinearRegression().fit(X_, simulado[objetivo])
    arbol = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_, simulado[objetivo])
    print(f"{objetivo:8s}  R² linear regression = {lineal.score(X_, simulado[objetivo]):.3f}   R² 2-level tree = {arbol.score(X_, simulado[objetivo]):.3f}")


The two-level tree (four steps) explains the U almost as well as the line; linear
regression explains practically none of the U. That is the underlying reason why
trees and their ensembles dominate tabular data: nobody has to tell them the shape.

## 5. What to take away

- The **information** of an event is $-\log_2 p$; **entropy** is the average information
  of a variable, and it measures how much we still do not know.
- The **information gain** (mutual information) of an attribute is how much entropy of the
  target disappears once we know it. It is the rule a tree uses to choose its questions.
- Entropy and variance are two measures of dispersion: variance needs numbers and a scale;
  entropy, only probabilities.
- Correlation and mutual information are two measures of relationship: correlation only sees straight lines;
  mutual information sees any shape. A low correlation does not always mean "there is no
  relationship".

## Exercises

**Exercise 1.** Compute the information gain of each attribute, but only with the 5
high-debt firms (`credito[credito.deuda == "alta"]`). Which attribute wins now? It is the
second question the tree will ask along that branch.

**Exercise 2.** In the U experiment, change the noise from 0.05 to 0.20 and repeat the table of
correlation and mutual information. How close does the U get to chance? And with 3,000 points instead of
300?

**Exercise 3.** Take the table of 219 firms from the course (it is in Notebook 2, section 4) and
compute, for each of the five ratios, its correlation with `impago` and its mutual information
with `impago` (with 5 bins). Do they rank the ratios the same way?


## Solutions


In [ ]:

# SOLUTION 1 -- the high-debt branch
alta = credito[credito.deuda == "alta"]
for a in ["tamano", "garantia", "historial"]:
    print(f"gain({a:9s}) in high debt = {ganancia(alta, 'paga', a):.3f} bits")
print("Collateral wins: among high-debt firms, having collateral perfectly separates the ones that pay.")


In [ ]:

# SOLUTION 2 -- more noise, more data
for n_, sd in [(300, 0.05), (300, 0.20), (3000, 0.05), (3000, 0.20)]:
    r = np.random.default_rng(3)
    x_ = r.uniform(0, 1, n_); e_ = r.normal(0, sd, n_)
    yu = 0.1 + 0.9 * (x_ - 0.5) ** 2 + e_; ya = 0.2 + 0.6 * r.uniform(0, 1, n_) + e_
    print(f"n = {n_:5d}, noise = {sd:.2f}:  U -> corr {np.corrcoef(x_, yu)[0,1]:6.3f}, MI {informacion_mutua(x_, yu):.3f} bits;"
          f"   chance -> corr {np.corrcoef(x_, ya)[0,1]:6.3f}, MI {informacion_mutua(x_, ya):.3f} bits")
print("With more noise the U gets closer to chance; with more data the chance floor drops and the U stands out better.")


In [ ]:

# SOLUTION 3 -- the ratios from the default table
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
sol = pd.DataFrame({
    "correlation with impago": [impago[r].corr(impago["impago"]) for r in ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]],
    "mutual information (5 bins)": [informacion_mutua(impago[r], impago["impago"].astype(float), 5) for r in ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]],
}, index=["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"])
print(sol.round(3))
print()
print("Note: 'impago' only has two values, so its 5 bins collapse to 2. Compare the ranking by |correlation| with the ranking by mutual information.")
